# 自定义中间件-Wrap-style hooks



In [2]:
from itertools import count

from langchain.agents.middleware.types import ResponseT
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langgraph.typing import ContextT

# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

## 1、wrap_model_call的使用

### 1、基于装饰器的实现

In [3]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, ExtendedModelResponse


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:

    request.messages[-1].content += "----> wrap_model_call_before <----"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "----> wrap_model_call_after <----"

    return response

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("你好！")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好！----> wrap_model_call_before <----
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？----> wrap_model_call_after <----


### 2、基于类的实现

In [8]:

from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self,
                        request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:

        request.messages[-1].content += "----> wrap_model_call_before <----"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "----> wrap_model_call_after <----"

        return response

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("你好！")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好！----> wrap_model_call_before <----
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？----> wrap_model_call_after <----


### 场景1：重试逻辑

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import time

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """自动重试失败的模型调用"""

    max_retries = 3

    for attempt in range(max_retries):
        try:
            print(f"✅ 尝试调用模型（第 {attempt + 1}/{max_retries} 次）")
            return handler(request)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ 所有重试失败：{e}")
                raise

            # 指数退避
            wait_time = 2 ** attempt

            print(f"⚠ 调用失败：{e}，{wait_time} 秒后重试")
            time.sleep(wait_time)

### 场景2：响应缓存

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import hashlib
import json

class ModelCache:
    """模型响应缓存"""
    def __init__(self):
        self.cache = {}

    def create_hook(self):
        @wrap_model_call
        def cache_model(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
            # 生成缓存键
            cache_key = hashlib.md5(
                json.dumps({
                    "messages": [str(m) for m in request.messages],
                    "system": str(request.system_message)
                }).encode()
            ).hexdigest()
            # 检查缓存
            if cache_key in self.cache:
                print("✅ 缓存命中！")
                return self.cache[cache_key]
            # 调用模型
            print("⏳ 缓存未命中，调用模型")
            response = handler(request)
            # 存入缓存
            self.cache[cache_key] = response
            return response
        return cache_model

# 使用
cache = ModelCache()
agent = create_agent(
    model=model,
    middleware=[cache.create_hook()]
)

### 场景3：修改系统提示


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import SystemMessage
from typing import Callable
from datetime import datetime

@wrap_model_call
def add_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """动态添加上下文信息到系统提示"""

    # 获取当前时间
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 构建新的系统消息
    original_content = request.system_message.content if request.system_message else ""
    new_content = f"""{original_content}

         当前时间：{current_time}
         用户位置：中国
         语言偏好：中文
         """

    # 创建新的系统消息
    new_system_message = SystemMessage(content=new_content)

    # 使用 override 方法修改请求（不可变，返回新请求对象）
    modified_request = request.override(system_message=new_system_message)

    return handler(modified_request)

## 2、wrap_tool_call的使用

### 2.1 基于装饰器的实现

In [15]:

from langchain_core.tools import tool
from langchain.agents import create_agent
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call

@tool
def get_weather(city : str, is_forcast : bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                             ) -> ToolMessage | Command[Any]:

    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{request}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{request}")

    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("杭州的天气如何？")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '杭州', 'is_forcast': False}
原始参数调用结果：ToolCallRequest(tool_call={'name': 'get_weather', 'args': {'city': '杭州', 'is_forcast': False}, 'id': 'call_9aac8817512942848c72cbd6', 'type': 'tool_call'}, tool=StructuredTool(name='get_weather', description='获取当日特定城市的天气\n\nArgs:\n    city: 城市名称\n    is_forcast: 是否包含明天的天气预报', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001F28EA53060>), state={'messages': [HumanMessage(content='杭州的天气如何？', additional_kwargs={}, response_metadata={}, id='c7346165-6430-44b6-a39d-cf3e6e4a4673'), AIMessage(content='我来帮您查询杭州今天的天气情况。\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 311, 'total_tokens': 358, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 311}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-71249d

### 2、基于类的实现

In [17]:

from langchain_core.tools import tool
from langchain.agents import create_agent
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call
from langchain.agents.middleware import AgentMiddleware

class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self,
                      request: ToolCallRequest,
                      handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                      ) -> ToolMessage | Command[Any]:

        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{request}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{request}")

        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        WrapToolCallMiddleware()
    ]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("杭州的天气如何？")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

原始参数：{'city': '杭州', 'is_forcast': False}
原始参数调用结果：ToolCallRequest(tool_call={'name': 'get_weather', 'args': {'city': '杭州', 'is_forcast': False}, 'id': 'call_5e86c10a74ab4c879df32b08', 'type': 'tool_call'}, tool=StructuredTool(name='get_weather', description='获取当日特定城市的天气\n\nArgs:\n    city: 城市名称\n    is_forcast: 是否包含明天的天气预报', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000001F28EA53060>), state={'messages': [HumanMessage(content='杭州的天气如何？', additional_kwargs={}, response_metadata={}, id='22e0aeba-034d-4b81-92f5-4866480114d1'), AIMessage(content='我来为您查询杭州今天的天气情况。\n\n', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 311, 'total_tokens': 358, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 311}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-4d145c

### 使用场景：用于监控、重试、修改工具执行。

比如：

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from typing import Callable
import time

@wrap_tool_call
def monitor_tool(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command]
) -> ToolMessage | Command:
    """监控工具执行时间和状态"""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call.get("args", {})
    print(f"🔍 开始执行工具：{tool_name}")
    print(f"   参数：{tool_args}")
    start_time = time.time()
    try:
        result = handler(request)
        elapsed = time.time() - start_time
        print(f"✅ 工具执行成功，耗时：{elapsed:.2f}秒")
        return result
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"❌ 工具执行失败：{e}，耗时：{elapsed:.2f}秒")
        raise